# 模块概述

WtRiskMonFact 是 WonderTrader 风险管理工厂模块，负责提供各种风险监控算法，实现组合盘的风险管理和资金保护。主要包括：
- 风控模块工厂管理和创建
- 基于回撤控制的风险监控算法
- 日内和多日双重风控保护机制
- 独立线程异步风控检查
- 仓位控制和盈利保护

1. **工厂层**（WtRiskMonFact）：
   - 实现 IRiskMonitorFact 接口，提供风控模块的创建、删除和管理功能
   - 支持风控模块的枚举和查询功能
   - 实现风控模块的生命周期管理
   - 提供C接口函数，支持动态库加载

2. **风控监控器层**（WtSimpleRiskMon）：
   - **简单风控监控器**：实现基于回撤控制的风险监控功能
   - **日内回撤风控**：监控日内从最高点的回撤幅度，超过阈值时降低仓位
   - **多日回撤风控**：监控多日最大动态权益的回撤幅度，超过阈值时清仓
   - **盈利保护**：当盈利达到一定比例后，启用回撤保护机制
   - **定时检查**：按照设定的时间间隔定期检查风险状况
   - **仓位控制**：通过设置数量倍数（vol_scale）来控制整体仓位比例

3. **风控算法特点**：
   - **日内回撤计算**：rate = (maxBal - curBal) * 100 / (maxBal - predynbal)
   - **多日回撤计算**：rate = (maxBal - curBal) * 100 / maxBal
   - **时间窗口控制**：使用日内分钟数计算时间差，避免午盘休息时间影响风控判断
   - **双重保护机制**：同时支持日内和多日风控，提供双重保护

4. **架构特点**：
   - **多线程设计**：使用独立线程执行风控检查，不阻塞主交易流程
   - **可配置参数**：支持通过配置文件设置各种风控参数，灵活适应不同策略需求
   - **精确时间控制**：使用毫秒级时间戳和睡眠机制，精确控制检查间隔
   - **详细日志记录**：记录每次检查的详细情况，便于分析和调试

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef factoryClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef monitorClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef contextClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IRiskMonitorFact["IRiskMonitorFact<br/>风控模块工厂接口<br/>• 创建风控模块<br/>• 删除风控模块<br/>• 枚举风控模块<br/>• 获取工厂名称"]:::interfaceClass
        WtRiskMonitor["WtRiskMonitor<br/>风控模块基类<br/>• 初始化接口<br/>• 启动接口<br/>• 停止接口<br/>• 获取名称接口"]:::interfaceClass
        WtPortContext["WtPortContext<br/>组合上下文接口<br/>• 资金信息查询<br/>• 交易状态查询<br/>• 仓位控制接口<br/>• 日志记录接口<br/>• 时间转换接口"]:::contextClass
    end

    %% 工厂层
    subgraph Factory["工厂层 - 风控模块工厂"]
        direction TB
        WtRiskMonFact["WtRiskMonFact<br/>风控模块工厂<br/>• 创建风控模块<br/>• 删除风控模块<br/>• 枚举风控模块<br/>• C接口导出"]:::factoryClass
    end

    %% 风控监控器层
    subgraph Monitors["风控监控器层 - 风险监控实现"]
        direction TB
        WtSimpleRiskMon["WtSimpleRiskMon<br/>简单风控监控器<br/>• 日内回撤风控<br/>• 多日回撤风控<br/>• 盈利保护机制<br/>• 定时检查机制<br/>• 仓位控制<br/>• 独立线程执行"]:::monitorClass
    end

    %% 继承关系
    WtRiskMonFact -.->|"实现"| IRiskMonitorFact
    WtSimpleRiskMon -.->|"继承"| WtRiskMonitor

    %% 工厂创建关系
    WtRiskMonFact -->|"创建"| WtSimpleRiskMon

    %% 风控监控器使用组合上下文
    WtSimpleRiskMon -->|"使用"| WtPortContext
    WtSimpleRiskMon -.->|"初始化"| WtPortContext

    %% 应用样式
    class WtRiskMonFact factoryClass
    class WtSimpleRiskMon monitorClass
    class IRiskMonitorFact,WtRiskMonitor interfaceClass
    class WtPortContext contextClass
```

# 风控模块工厂 WtRiskMonFact.h/cpp
```cpp
class WtRiskMonFact : public IRiskMonitorFact
```
负责创建、管理和销毁各种风控监控器实例。

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称
 * 
 * 返回当前工厂的名称标识字符串，用于区分不同的风控模块工厂。
 * 
 * @return 返回工厂名称常量"WtRiskMonFact"
 */
const char* WtRiskMonFact::getName()
{
	return FACT_NAME;
}
```

## 枚举所有可用的风控模块 enumRiskMonitors
```cpp
/**
 * @brief 枚举所有可用的风控模块
 * 
 * 遍历当前工厂支持的所有风控模块类型，通过回调函数逐一通知调用者。
 * 当前工厂支持"SimpleRiskMon"（简单风控监控器）模块。
 * 
 * @param cb 枚举回调函数，用于接收每个风控模块的信息
 * 
 * 回调函数参数说明：
 * - factName: 工厂名称，值为"WtRiskMonFact"
 * - unitName: 风控模块名称，当前为"SimpleRiskMon"
 * - isLast: 是否为最后一个模块，当前为true（只有一个模块）
 */
void WtRiskMonFact::enumRiskMonitors(FuncEnumRiskMonCallback cb)
{
	//cb(FACT_NAME, "WtSimpExeUnit", false);
	cb(FACT_NAME, "SimpleRiskMon", true); // 调用回调函数，通知SimpleRiskMon模块，isLast为true表示这是最后一个模块
}
```

## 根据名称创建风控监控器实例 createRiskMonotor
```cpp
/**
 * @brief 根据名称创建风控监控器实例
 * 
 * 根据传入的风控模块名称，创建对应的风控监控器对象。
 * 当前支持创建"SimpleRiskMon"模块，其他名称返回NULL。
 * 
 * @param name 风控模块名称字符串
 * @return 成功返回风控监控器对象指针，失败返回NULL
 * 
 * 支持的模块：
 * - "SimpleRiskMon": 创建WtSimpleRiskMon实例，实现基础的日内和多日回撤风控
 * 
 * 注意事项：
 * - 返回的对象需要调用者负责管理生命周期
 * - 删除对象时应使用deleteRiskMonotor方法，确保安全删除
 */
WtRiskMonitor* WtRiskMonFact::createRiskMonotor(const char* name)
{
	if (strcmp(name, "SimpleRiskMon") == 0)  // 比较名称是否匹配"SimpleRiskMon"
		return new WtSimpleRiskMon();  // 创建简单风控监控器实例并返回
	return NULL;
}
```

## 删除风控监控器实例 deleteRiskMonotor
```cpp
/**
 * @brief 删除风控监控器实例
 * 
 * 安全地销毁传入的风控监控器对象，释放其占用的资源。
 * 在删除前会进行多重检查，确保安全删除：
 * 1. 检查对象指针是否为NULL
 * 2. 检查对象是否属于当前工厂创建
 * 
 * @param unit 要删除的风控监控器对象指针
 * @return 删除成功返回true，失败返回false
 *
 * 安全机制：
 * - 防止误删：只删除属于当前工厂创建的对象
 * - 空指针保护：NULL指针直接返回成功，避免崩溃
 */
bool WtRiskMonFact::deleteRiskMonotor(WtRiskMonitor* unit)
{
	if (unit == NULL)
		return true;
	if (strcmp(unit->getFactName(), FACT_NAME) != 0)
		return false;
	delete unit; // 工厂名称匹配，安全删除对象
	return true;
}
```

# 简单风控监控器 WtSimpRiskMon.h/cpp
```cpp
class WtSimpleRiskMon : public WtRiskMonitor
```
实现了基于回撤控制的风险监控功能。该风控监控器通过独立线程持续监控组合盘的资金状况，当检测到风险超过预设阈值时，自动采取降低仓位等风控措施，保护组合资金安全。

## 成员
- **线程管理**
  - `ThreadPtr _thrd`：风控检查线程智能指针
    - typedef std::shared_ptr\<std::thread\> ThreadPtr：线程智能指针类型，用于管理线程对象的生命周期
  - `bool _stopped`：停止标志，true 表示风控监控已停止，false 表示正在运行
  - `bool _limited`：仓位限制标志，true 表示已触发仓位限制，false 表示未限制

- **时间管理**
  - `uint64_t _last_time`：上次检查时间戳（毫秒），用于计算时间间隔

- **风控配置参数**
  - `uint32_t _calc_span`：计算时间间隔，单位：秒，风控检查的执行频率
  - `uint32_t _risk_span`：回撤比较时间，单位：分钟，日内回撤检查的时间窗口
  - `double _basic_ratio`：基础盈利率，单位：百分比，触发回撤保护的盈利阈值（如 101 表示 101%）
  - `double _risk_scale`：风险控制系数，范围：0-1，触发风控后的仓位比例（如 0.3 表示 30% 仓位）
  - `double _inner_day_fd`：日内高点回撤边界，单位：百分比，日内回撤触发阈值（如 80 表示 80%）
  - `bool _inner_day_active`：日内风控启用标志，true 表示启用日内回撤风控，false 表示禁用
  - `double _multi_day_fd`：多日高点回撤边界，单位：百分比，多日回撤触发阈值（如 20 表示 20%）
  - `bool _multi_day_active`：多日风控启用标志，true 表示启用多日回撤风控，false 表示禁用
  - `double _base_amount`：基础资金规模，单位：金额，用于计算权益比例的基础资金

- **基类成员（继承自 WtRiskMonitor）**
  - `WtPortContext* _ctx`：组合上下文对象指针，提供资金数据、交易状态、日志记录等功能

## 接口实现—WtRiskMonitor

### 获取风控监控器名称 getName
```cpp
/**
 * @brief 获取风控监控器名称
 * 
 * 返回当前风控监控器的名称标识，用于区分不同的风控模块。
 * 
 * @return 返回"WtSimpleRiskMon"字符串常量
 */
const char* WtSimpleRiskMon::getName()
{
	return "WtSimpleRiskMon";
}
```

### 获取所属工厂名称 getFactName
```cpp
/**
 * @brief 获取所属工厂名称
 * 
 * 返回创建当前风控监控器的工厂名称，用于工厂管理和对象归属判断。
 * 
 * @return 返回工厂名称常量FACT_NAME的值（"WtRiskMonFact"）
 */
const char* WtSimpleRiskMon::getFactName()
{
	return FACT_NAME;  // 返回工厂名称常量
}
```

### 初始化风控监控器 init

### 启动风控监控 run
启动一个**独立的后台线程**，用于持续、异步地监控组合账户的资金风险状态。该函数不阻塞主线程，而是通过死循环配合精细的睡眠机制，定期执行 *日内高点回撤* 和 *多日最大权益回撤* 的双重检查。
1. **线程启动检查**
   * 首先检查 `_thrd` 是否已经存在。
   * 如果已启动，直接返回，防止重复创建线程。
2. **创建并执行监控线程**
   * 创建一个新的 `std::thread`，并在其中运行一个 Lambda 表达式。
   * **主循环**：只要 `_stopped` 标志为 `false`，线程就会一直运行。
3. **交易状态校验**
   * 在每次检查前，调用 `_ctx->isInTrading()`。
   * 只有在**交易时段内**且上下文有效时，才读取资金数据 `fs` 进行计算。
     * `fundInfo = _ctx->getFundInfo()`
     * `fs = fundInfo->fundInfo()`
4. **核心风控逻辑 A：日内回撤风控**
   * **前提**：开关 `_inner_day_active` 开启，且当日最大动态权益 `fs._max_dyn_bal` 已初始化。
   * **数据准备**：
     * `predynbal`：上日静态权益 `fundInfo->predynbalance()` + 基础资金 `_base_amount`
     * `maxBal`：当日最大动态权益 `fs._max_dyn_bal` + 基础资金 `_base_amount`
     * `curBal`：当前动态权益（当前资金余额 `fs._balance` + 浮动盈亏 `fs._dynprofit`） + 基础资金 `_base_amount`
   * **触发前提（盈利保护）**：只有当 `maxBal > _basic_ratio * predynbal / 100.0` 时，才激活回撤检查。
   * **计算 *当日盈利回撤比例***：
     * `rate = (当日最大动态权益 - 当前动态权益) / (当日最大动态权益 - 上日静态权益 - 基础资金)`
   * **条件**：
     * `回撤率 rate >= _inner_day_fd` && 
     * `当前时间在交易时段内分钟数 - 当日最大动态权益出现时间在交易时段内分钟数 <= _risk_span` 
     * 尚未触发过限制 `!_limited`
   * **执行动作**：
     * **降仓**：调用 `_ctx->setVolScale(_risk_scale)`（例如降低到30%仓位）。
     * **锁定**：标记 `_limited = true`，防止重复触发。
5. **核心风控逻辑 B：多日回撤风控**
   * **前提**：开关 `_multi_day_active` 开启，且历史最大动态权益 `fs._max_md_dyn_bal` 已初始化。
   * **数据准备**：
     * `maxBal`：多日最大动态权益 `fs._max_md_dyn_bal` + 基础资金 `_base_amount`
     * `curBal`：当前动态权益（当前资金余额 `fs._balance` + 浮动盈亏 `fs._dynprofit`） + 基础资金 `_base_amount`
     * 获取历史记录中的多日最大动态权益 (`_max_md_dyn_bal`)。
   * **回撤检查触发前提**：`curBal < maxBal`
   * **回撤计算**：`rate = (多日最大动态权益 - 当前动态权益) * 100 / (多日最大动态权益 + 基础资金)`
   * **条件**：
     * `回撤率 rate >= _multi_day_fd`
   * **执行动作**：
     * **清仓**：调用 `_ctx->setVolScale(0.0)`（强制平掉所有仓位）。
6. **精细化休眠机制**
   * 为了保证线程能响应 `stop()` 指令，函数**没有**直接使用长时间的 `sleep`。
   * 记录 `_last_time` 为当前时间
   * **循环微休眠**：
     * 在一个小循环中，每次休眠 **2毫秒**。
     * 检查 `now - _last_time >= _calc_span` (检查间隔，如1秒)。
   * **优势**：如果在休眠期间用户调用了 `stop()` 修改了 `_stopped` 标志，线程可以立刻退出，而不需要等待整个 `_calc_span` 结束。

```cpp
/**
 * @brief 启动风控监控
 */
void WtSimpleRiskMon::run()
```

### 停止风控监控 stop
```cpp
/**
 * @brief 停止风控监控
 * 
 * 停止风控检查线程，等待线程安全退出。
 * 设置_stopped标志为true，通知线程退出，然后等待线程结束。

 * 注意事项：
 * - join方法会阻塞当前线程，直到目标线程结束
 */
void WtSimpleRiskMon::stop()
{
	_stopped = true;
	if (_thrd)
		_thrd->join();
}
```